In [ ]:
"""
Week 2 - Day 7
Final Summary + Mid Review Prep
================================
Complete Week 2 wrap up demonstrating
all deliverables for mid review.

Week 2 Achievements:
✅ DQN Neural Network (PyTorch)
✅ Experience Replay Buffer
✅ Target Network
✅ Epsilon Greedy Strategy
✅ 1000 Season Simulation
✅ Statistical Proof
✅ Price Trajectory Dashboard
✅ Deadline Discounting Proved

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
import os
import sys

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..", "..")
)

SRC_DIR = os.path.join(PROJECT_ROOT, "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Project Root :", PROJECT_ROOT)
print("SRC Exists   :", os.path.exists(SRC_DIR))

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)
from agents.dqn.dqn_agent import DQNAgent
from agents.q_learning_agent import (
    QLearningAgent,
    QL_CONFIG
)
from agents.baseline_agents import (
    FixedPriceAgent,
    TimedPricingAgent,
    DemandBasedAgent,
    LinearDecayAgent
)
from agents.agent_registry import (
    create_all_baseline_agents
)
from utils.evaluator import evaluate_agent
from config import DQN, PROJECT_INFO

plt.style.use('seaborn-v0_8')
print("✅ Week 2 Final Summary loaded!")
print(f"\nProject: {PROJECT_INFO['name']}")
print(f"Program: {PROJECT_INFO['program']}")

In [ ]:
env = DynamicPricingEnv()

# Train DQN
print("Training DQN (2000 episodes)...")
dqn_agent = DQNAgent(env, DQN)
dqn_agent.train(n_episodes=2000, verbose=False)
dqn_eval  = dqn_agent.evaluate(n_episodes=100)
print(f"✅ DQN: ${dqn_eval['mean_revenue']:.0f}")

# Train Q-Learning
print("\nTraining Q-Learning (3000 episodes)...")
ql_agent = QLearningAgent(env, QL_CONFIG)
ql_agent.train(n_episodes=3000, verbose=False)
ql_eval  = ql_agent.evaluate(n_episodes=100)
print(f"✅ Q-Learning: ${ql_eval['mean_revenue']:.0f}")

# Evaluate baselines
print("\nEvaluating baselines...")
baselines = create_all_baseline_agents(env)
bl_results = {}
for agent in baselines:
    df = evaluate_agent(agent, n_episodes=100)
    bl_results[agent.name] = (
        df['total_revenue'].mean()
    )
    print(f"  {agent.name}: "
          f"${bl_results[agent.name]:.0f}")

In [ ]:
all_results = {
    **bl_results,
    'Q-Learning' : ql_eval['mean_revenue'],
    'DQN 🤖'     : dqn_eval['mean_revenue'],
}

ranked = sorted(
    all_results.items(),
    key=lambda x: x[1],
    reverse=True
)

best_bl = max(bl_results.values())
dqn_rev = dqn_eval['mean_revenue']
improvement = (dqn_rev - best_bl) / best_bl * 100

print("\n=== FINAL WEEK 2 RANKINGS ===\n")
medals = ['🥇', '🥈', '🥉',
          '4️⃣', '5️⃣', '6️⃣', '7️⃣']
for i, (name, rev) in enumerate(ranked):
    print(f"  {medals[i]} {name:<20}: ${rev:.0f}")

print(f"\n  DQN vs Best Baseline: {improvement:+.1f}%")

In [ ]:
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig)

colors = ['gold', 'coral', 'steelblue',
          'green', 'purple', 'orange', 'pink']

# ── Chart 1: Revenue Comparison ──
ax1 = fig.add_subplot(gs[0, :2])
names    = [n for n, _ in ranked]
revenues = [r for _, r in ranked]
bar_colors = [
    'gold' if '🤖' in n
    else 'coral' if 'Q-Learning' in n
    else 'steelblue'
    for n in names
]
bars = ax1.bar(
    names, revenues,
    color=bar_colors,
    edgecolor='black',
    width=0.6
)
ax1.set_title(
    'Final Rankings — Mean Revenue',
    fontweight='bold', fontsize=13
)
ax1.set_ylabel('Mean Revenue ($)')
ax1.set_xticklabels(
    names, rotation=15, fontsize=9
)
for bar, val in zip(bars, revenues):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        val + 20,
        f'${val:.0f}',
        ha='center', fontsize=9,
        fontweight='bold'
    )

# ── Chart 2: DQN Training Curve ──
ax2 = fig.add_subplot(gs[0, 2])
smooth = pd.Series(
    dqn_agent.episode_rewards
).rolling(window=100).mean()
ax2.plot(
    dqn_agent.episode_rewards,
    alpha=0.2, color='steelblue',
    linewidth=0.5
)
ax2.plot(
    smooth, color='red',
    linewidth=2.5,
    label='100-ep avg'
)
ax2.set_title(
    'DQN Training Curve',
    fontweight='bold'
)
ax2.set_xlabel('Episode')
ax2.set_ylabel('Revenue ($)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# ── Chart 3: Price Trajectory DQN ──
ax3 = fig.add_subplot(gs[1, :2])
# Run 5 episodes and overlay
for ep in range(5):
    state, _ = env.reset(seed=ep)
    prices   = []
    done     = False
    while not done:
        action = dqn_agent.select_action(
            state, training=False
        )
        prices.append(PRICE_LEVELS[action])
        state, _, term, trunc, _ = (
            env.step(action)
        )
        done = term or trunc
    ax3.plot(
        prices,
        alpha=0.6, linewidth=2,
        marker='o', markersize=3
    )

ax3.set_title(
    'DQN Price Trajectories — 5 Episodes\n'
    'Shows Deadline Discounting!',
    fontweight='bold'
)
ax3.set_xlabel('Day')
ax3.set_ylabel('Price ($)')
ax3.set_ylim([0, 350])
ax3.grid(True, alpha=0.3)
ax3.axvspan(
    25, 30, alpha=0.1,
    color='red', label='Deadline Zone'
)
ax3.legend()

# ── Chart 4: Epsilon Decay ──
ax4 = fig.add_subplot(gs[1, 2])
ax4.plot(
    dqn_agent.episode_epsilons,
    color='green', linewidth=2
)
ax4.set_title(
    'Epsilon Decay\nExplore → Exploit',
    fontweight='bold'
)
ax4.set_xlabel('Episode')
ax4.set_ylabel('Epsilon (ε)')
ax4.grid(True, alpha=0.3)

# ── Chart 5: Week Progress ──
ax5 = fig.add_subplot(gs[2, :])
weeks  = [
    'W1D1\nMDP+Env',
    'W1D2\nBaselines',
    'W1D3\nQ-Learning',
    'W1D4\nQ-Analysis',
    'W1D5\nWrap Up',
    'W2D1\nDQN Arch',
    'W2D2\nDQN Train',
    'W2D3\n1000 Sim',
    'W2D4\nTrajectory',
    'W2D5\nAnalysis',
    'W2D6\nRefactor',
    'W2D7\nFinal'
]
progress = [
    8, 17, 25, 33, 42,
    50, 58, 67, 75, 83,
    92, 100
]
ax5.plot(
    weeks, progress,
    marker='o', linewidth=2.5,
    color='steelblue',
    markersize=10,
    markerfacecolor='gold'
)
ax5.fill_between(
    range(len(weeks)),
    progress,
    alpha=0.2, color='steelblue'
)
for i, (w, p) in enumerate(
    zip(weeks, progress)
):
    ax5.annotate(
        f'{p}%',
        xy=(i, p),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
        fontsize=8,
        fontweight='bold'
    )
ax5.set_title(
    'Project Progress Timeline\n'
    'Week 1 + Week 2',
    fontweight='bold', fontsize=12
)
ax5.set_ylabel('Completion %')
ax5.set_ylim(0, 115)
ax5.grid(True, alpha=0.3)

plt.suptitle(
    'Week 2 Final Dashboard\n'
    'RL Dynamic Pricing — Project 2',
    fontsize=15, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    '../results/week2_final_dashboard.png',
    bbox_inches='tight', dpi=150
)
plt.show()
print("✅ Final dashboard saved!")

In [ ]:
# Prove deadline discounting
print("=== PROVING DQN BEHAVIORS ===\n")

early_prices  = []
urgent_prices = []
high_inv      = []
low_inv       = []

for ep in range(100):
    state, _ = env.reset(seed=ep)
    done     = False
    while not done:
        action = dqn_agent.select_action(
            state, training=False
        )
        price    = PRICE_LEVELS[action]
        days     = int(state[1])
        inv      = int(state[0])

        if days >= 20:
            early_prices.append(price)
        elif days <= 5:
            urgent_prices.append(price)

        if inv >= 40:
            high_inv.append(price)
        elif inv <= 10:
            low_inv.append(price)

        state, _, term, trunc, _ = (
            env.step(action)
        )
        done = term or trunc

avg_early  = np.mean(early_prices)
avg_urgent = np.mean(urgent_prices)
avg_high   = np.mean(high_inv)
avg_low    = np.mean(low_inv)
drop_pct   = (avg_early-avg_urgent)/avg_early*100
scar_pct   = (avg_low-avg_high)/avg_high*100

print(f"  BEHAVIOR 1 — DEADLINE DISCOUNTING:")
print(f"  Early price  : ${avg_early:.0f}")
print(f"  Urgent price : ${avg_urgent:.0f}")
print(f"  Price drop   : -{drop_pct:.1f}%")
if avg_urgent < avg_early:
    print(f"  ✅ PROVED!")

print(f"\n  BEHAVIOR 2 — SCARCITY PRICING:")
print(f"  High inv price: ${avg_high:.0f}")
print(f"  Low inv price : ${avg_low:.0f}")
print(f"  Price premium : +{scar_pct:.1f}%")
if avg_low > avg_high:
    print(f"  ✅ PROVED!")

In [ ]:
print("╔══════════════════════════════════════════╗")
print("║      WEEK 2 FINAL SUMMARY                ║")
print("╠══════════════════════════════════════════╣")
print("║  WEEK 1 DELIVERABLES:         ✅         ║")
print("║  → MDP + Custom Gym Env                  ║")
print("║  → 5 Baseline Agents                     ║")
print("║  → Q-Learning (5000 eps)                 ║")
print("║  → Q-Table Analysis                      ║")
print("╠══════════════════════════════════════════╣")
print("║  WEEK 2 DELIVERABLES:         ✅         ║")
print("║  → DQN Neural Network                    ║")
print("║  → Experience Replay Buffer              ║")
print("║  → Target Network                        ║")
print("║  → Epsilon Greedy Strategy               ║")
print("║  → 1000 Season Simulation                ║")
print("║  → Statistical Proof (p<0.05)            ║")
print("║  → Price Trajectory Dashboard            ║")
print("║  → Deadline Discounting Proved           ║")
print("║  → Scarcity Pricing Proved               ║")
print("╠══════════════════════════════════════════╣")
print(f"║  DQN vs Best Baseline: "
      f"{improvement:+.1f}%"
      f"{'':<19} ║")
print("╠══════════════════════════════════════════╣")
print("║  GITHUB:                                 ║")
print("║  Commits : 14+ days ✅                   ║")
print("║  Issues  : All closed ✅                 ║")
print("╠══════════════════════════════════════════╣")
print("║  🎯 MID REVIEW READY!                    ║")
print("╚══════════════════════════════════════════╝")